In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import os
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
import time
from sklearn.metrics import f1_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

# ============================================================
# CONFIG
# ============================================================
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

BATCH_SIZE = 32
NUM_EPOCHS = 10
LR = 1e-4
WEIGHT_DECAY = 1e-4
T_MAX = NUM_EPOCHS  # cosine schedule decays fully across however many epochs we actually run
ETA_MIN = 1e-6
LABEL_MAP = {"LumA": 0, "LumB": 1, "Her2": 2, "Basal": 3}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# DATA
# ============================================================
csv_path = "/kaggle/input/datasets/anishapanja/bc-xai-patches-split-2000/patches_with_split_2000.csv"
df = pd.read_csv(csv_path)

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)
print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)}")

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02, p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.1),
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

eval_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

class PatchDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["patch_path"]).convert("RGB")
        img = np.array(img)
        img = self.transform(image=img)["image"]
        label = LABEL_MAP[row["subtype_clean"]]
        return img, label

train_dataset = PatchDataset(train_df, train_transform)
val_dataset = PatchDataset(val_df, eval_transform)
test_dataset = PatchDataset(test_df, eval_transform)

class_counts = train_df["subtype_clean"].map(LABEL_MAP).value_counts().reindex([0, 1, 2, 3])
assert list(class_counts.index) == [0, 1, 2, 3], "Class count index order is wrong!"
assert class_counts.isna().sum() == 0, "A class is missing from train split!"

class_weights = 1.0 / class_counts.values
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

train_labels = train_df["subtype_clean"].map(LABEL_MAP).values
sample_weights = class_weights[train_labels]
sample_weights_tensor = torch.tensor(sample_weights, dtype=torch.float32)

sampler = WeightedRandomSampler(weights=sample_weights_tensor, num_samples=len(sample_weights_tensor), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"train_loader batches: {len(train_loader)}")
print(f"val_loader batches: {len(val_loader)}")
print(f"test_loader batches: {len(test_loader)}")

# ============================================================
# MODEL
# ============================================================
model = timm.create_model("tf_efficientnet_b4", pretrained=True, num_classes=4)
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_MAX, eta_min=ETA_MIN)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

print(f"Model loaded. Params: {sum(p.numel() for p in model.parameters()):,}")

# ============================================================
# TRAIN / EVAL FUNCTIONS
# ============================================================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        all_preds.extend(outputs.argmax(dim=1).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, macro_f1

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, macro_f1

# ============================================================
# TRAINING LOOP
# ============================================================
best_val_f1 = -1.0
best_epoch = -1
history = []

print(f"\nStarting training: {NUM_EPOCHS} epochs")
overall_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()
    train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_f1 = evaluate(model, val_loader, criterion, device)
    scheduler.step()
    epoch_time = time.time() - epoch_start

    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch}/{NUM_EPOCHS} | train_loss={train_loss:.4f} train_f1={train_f1:.4f} | "
          f"val_loss={val_loss:.4f} val_f1={val_f1:.4f} | lr={current_lr:.2e} | time={epoch_time/60:.1f}min")

    history.append({"epoch": epoch, "train_loss": train_loss, "train_f1": train_f1,
                     "val_loss": val_loss, "val_f1": val_f1, "lr": current_lr})

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "efficientnet_b4_best.pth"))
        print(f"  -> New best val_f1={val_f1:.4f}, saved checkpoint.")

total_time = time.time() - overall_start
print(f"\nTraining complete. Total time: {total_time/3600:.2f} hours")
print(f"Best val_f1={best_val_f1:.4f} at epoch {best_epoch}")

pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, "efficientnet_b4_history.csv"), index=False)

# ============================================================
# FINAL TEST SET EVALUATION (using best checkpoint)
# ============================================================
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "efficientnet_b4_best.pth")))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs)

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

precision, recall, f1, support = precision_recall_fscore_support(all_labels, all_preds, labels=[0,1,2,3])
macro_f1_test = f1_score(all_labels, all_preds, average="macro")

print("\n=== TEST SET RESULTS (EfficientNet-B4) ===")
for i in range(4):
    cls_name = INV_LABEL_MAP[i]
    print(f"{cls_name}: precision={precision[i]:.4f} recall={recall[i]:.4f} f1={f1[i]:.4f} support={support[i]}")
print(f"\nMacro F1: {macro_f1_test:.4f}")

auc_scores = {}
for i in range(4):
    y_true_binary = (all_labels == i).astype(int)
    y_score = all_probs[:, i]
    auc_scores[INV_LABEL_MAP[i]] = roc_auc_score(y_true_binary, y_score)
print("\nAUC-ROC (one-vs-rest):")
for cls_name, auc in auc_scores.items():
    print(f"  {cls_name}: {auc:.4f}")

cm = confusion_matrix(all_labels, all_preds, labels=[0,1,2,3])
print("\nConfusion Matrix (rows=true, cols=pred, order=LumA,LumB,Her2,Basal):")
print(cm)

# Save test predictions for later XAI/TIL analysis (Tasks 3-5 need to know which test row is which)
test_results_df = test_df.copy()
test_results_df["pred_label"] = all_preds
test_results_df["pred_subtype"] = [INV_LABEL_MAP[p] for p in all_preds]
for i in range(4):
    test_results_df[f"prob_{INV_LABEL_MAP[i]}"] = all_probs[:, i]
test_results_df.to_csv(os.path.join(OUTPUT_DIR, "efficientnet_b4_test_predictions.csv"), index=False)

print(f"\nAll outputs saved to {OUTPUT_DIR}/:")
for f in os.listdir(OUTPUT_DIR):
    print(" -", f)

Device: cuda
train=219751 val=44731 test=44362
train_loader batches: 6868
val_loader batches: 1398
test_loader batches: 1387


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

Model loaded. Params: 17,555,788

Starting training: 10 epochs
Epoch 1/10 | train_loss=0.6259 train_f1=0.6167 | val_loss=2.0919 val_f1=0.3083 | lr=9.76e-05 | time=55.3min
  -> New best val_f1=0.3083, saved checkpoint.
Epoch 2/10 | train_loss=0.2890 train_f1=0.8120 | val_loss=2.4651 val_f1=0.2954 | lr=9.05e-05 | time=53.6min
Epoch 3/10 | train_loss=0.1880 train_f1=0.8718 | val_loss=2.5882 val_f1=0.3200 | lr=7.96e-05 | time=53.6min
  -> New best val_f1=0.3200, saved checkpoint.
Epoch 4/10 | train_loss=0.1340 train_f1=0.9075 | val_loss=2.8759 val_f1=0.3302 | lr=6.58e-05 | time=53.6min
  -> New best val_f1=0.3302, saved checkpoint.
Epoch 5/10 | train_loss=0.0977 train_f1=0.9302 | val_loss=2.8906 val_f1=0.3500 | lr=5.05e-05 | time=53.9min
  -> New best val_f1=0.3500, saved checkpoint.
Epoch 6/10 | train_loss=0.0739 train_f1=0.9469 | val_loss=3.5425 val_f1=0.3510 | lr=3.52e-05 | time=53.6min
  -> New best val_f1=0.3510, saved checkpoint.
Epoch 7/10 | train_loss=0.0556 train_f1=0.9592 | val_l